# TSBD 30-case paired bootstrap verification
Use only genuine case-level predictions. Required columns: `image_id,true_label,yolo_only_pred,yolo_ontology_pred`.


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, accuracy_score
INPUT=Path('tsbd_30_predictions.csv'); B=2000; SEED=42
df=pd.read_csv(INPUT)
required=['image_id','true_label','yolo_only_pred','yolo_ontology_pred']
assert all(c in df.columns for c in required)
assert len(df)==30
assert not df[required].isna().any().any()
for c in ['true_label','yolo_only_pred','yolo_ontology_pred']:
    if not pd.api.types.is_numeric_dtype(df[c]): df[c]=df[c].astype(str).str.strip()
labels=sorted(set(df.true_label)|set(df.yolo_only_pred)|set(df.yolo_ontology_pred),key=str)
y=df.true_label.to_numpy(); a=df.yolo_only_pred.to_numpy(); b=df.yolo_ontology_pred.to_numpy()
mf=lambda t,p:f1_score(t,p,labels=labels,average='macro',zero_division=0)
rng=np.random.default_rng(SEED); ba=[]; bb=[]; dd=[]
for _ in range(B):
    idx=rng.integers(0,len(y),len(y)); fa=mf(y[idx],a[idx]); fb=mf(y[idx],b[idx]); ba.append(fa); bb.append(fb); dd.append(fb-fa)
ba=np.asarray(ba);bb=np.asarray(bb);dd=np.asarray(dd);lo,hi=np.percentile(dd,[2.5,97.5]);p=min(1.0,2*min(np.mean(dd<=0),np.mean(dd>=0)))
summary=pd.DataFrame([{'n_cases':len(df),'bootstrap_resamples':B,'seed':SEED,'yolo_only_point_macro_f1':mf(y,a),'yolo_ontology_point_macro_f1':mf(y,b),'yolo_only_accuracy':accuracy_score(y,a),'yolo_ontology_accuracy':accuracy_score(y,b),'bootstrap_mean_macro_f1_yolo_only':ba.mean(),'bootstrap_mean_macro_f1_yolo_ontology':bb.mean(),'bootstrap_mean_difference_ontology_minus_yolo':dd.mean(),'ci95_low':lo,'ci95_high':hi,'two_sided_empirical_bootstrap_p':p}])
display(summary.T)
summary.to_csv('bootstrap_30case_results.csv',index=False)
pd.DataFrame({'resample':range(1,B+1),'macro_f1_yolo_only':ba,'macro_f1_yolo_ontology':bb,'diff_ontology_minus_yolo':dd}).to_csv('bootstrap_30case_resamples.csv',index=False)
print('Saved output CSV files.')
